# The Lab — Kaggle Experiment

From a Kaggle dataset reference to trained models: ingest → context pack → clean → agent proposal → approve & run → results.

Companion script: `examples/kaggle_experiment.py`. Uses the Python API only; every step is also available via the UI and the HTTP API.

## 1. Ingest from Kaggle (network)

In [ ]:
from thelab.ide.kaggle_api import (
    build_context_pack,
    fetch_kaggle_page_context,
    get_dataset_context,
    ingest_kaggle_dataset,
)

SLUG = "erfan4524/e-commerce-sales-data-analysis-and-eda"
ingestion = ingest_kaggle_dataset(SLUG)
ingestion["dataset_id"], ingestion["profile"]["rows"], ingestion["profile"]["columns"]

## 2. The dataset's own documentation (context pack)

In [ ]:
page = fetch_kaggle_page_context(SLUG)
pack = build_context_pack(SLUG, ingestion, page)

print((pack.get("description_markdown") or "")[:600])
assert get_dataset_context(ingestion["dataset_id"]) is not None

## 3. Clean (deterministic policy)

In [ ]:
from thelab.ide.cleaning import clean_dataset

TARGET = "Sales"
cleaned = clean_dataset(ingestion["dataset_id"], target=TARGET)
print(cleaned["dataset_id"], f"{cleaned['rows']} rows x {cleaned['columns']} cols")
for action in cleaned["cleaning_report"]["actions"]:
    print("-", action)

## 4. Agent proposal

In [ ]:
import asyncio
from thelab.agents.mock import MockProvider
from thelab.agents.worker import WorkerAgent
from thelab.ide.datasets import dataset_id_to_relative_path

worker = WorkerAgent(provider=MockProvider([]), servers=[], proposals_dir="proposals")
proposal = asyncio.run(worker.propose(
    goal="Predict order Sales (Kaggle e-commerce dataset)",
    dataset=dataset_id_to_relative_path(cleaned["dataset_id"]),
    target=TARGET,
    model_grid=["random_forest_regressor", "hist_gradient_boosting_regressor", "ridge"],
    seeds=[42],
))
print(proposal.proposal_id, proposal.task_type, proposal.model_grid)

## 5. Approve + run

In [ ]:
from thelab.ide.proposals_api import approve_and_run_proposal

outcome = approve_and_run_proposal(proposal.proposal_id, principal="kaggle_experiment")
print(outcome["status"], "| completed:", outcome["completed"])

## 6. Results

In [ ]:
import json, os
from pathlib import Path

runs_root = Path(os.environ.get("THELAB_RUNS_ROOT", "runs"))
for entry in outcome["results"]:
    metrics = json.loads((runs_root / entry["run_id"] / "metrics.json").read_text())
    print(f"{entry['model']}: R2={metrics['test_r2']:.4f} RMSE={metrics['test_rmse']:.2f} ({entry['status']})")

## 7. The honest finding

`Sales` is a **computed column**: `Sales = UnitPrice × Quantity × (1 − Discount/100)` — verified against the raw data. Near-perfect R² is arithmetic, not modeling skill. The Lab's context/EDA workflow is what surfaces this; every iteration (including the rejections that led here) is a traceable run.